# NB15 — Verifier prototype on the stage-1 model (SPEC T3, go/no-go)

Builds S/D candidate tables from the NB14 stage-1 caches, trains a LightGBM verifier on fold A + fold B +
the Iqra dev slice (all out-of-fold for stage-1), picks τ on Extra-dev, and decides **go** if
verifier + stage-1 beats B0 by ≥ 0.02 F1 on Extra-dev with the 90% paired sentence-bootstrap interval
excluding 0. **CPU only.** Never touches QuranMB.v2 or IqraEval_Test_GT.

**Run settings:** Accelerator *None*, Internet *On*.
Add data: `iqra-cache`, the NB11 output, the NB14 output (`post_stage1_*.pt`). Secrets: `HF_TOKEN`.

**After the run:** Save Version. Paste back everything printed by the last cell (**PASTE BACK**).


In [ ]:
# ---- Shared install cell (identical in nb11-nb16) --------------------------------
# pip install git+... does not fetch submodules, and the official scorer is a pinned
# submodule, so clone recursively and install from the clone (SPEC decision 21).
import os, subprocess, sys
REPO_URL = "https://github.com/markeet04/Quran_phenome.git"
REPO_DIR = "/tmp/Quran_phenome"          # outside /kaggle/working: not saved as output
BRANCH = "main"

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")   # only if the repo is private
except Exception:
    pass
url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL

def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode:
        msg = r.stderr if not token else r.stderr.replace(token, "***")
        raise RuntimeError(f"git {args[0]} failed: {msg}")
    return r.stdout.strip()

if not os.path.isdir(REPO_DIR):
    _git("clone", "--recursive", "-q", "-b", BRANCH, url, REPO_DIR)
else:   # a re-run in the same session: move the existing clone to the latest commit
    _git("-C", REPO_DIR, "fetch", "-q", url, BRANCH)
    _git("-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD")
_git("-C", REPO_DIR, "submodule", "update", "--init", "--recursive", "-q")
REPO_COMMIT = _git("-C", REPO_DIR, "rev-parse", "HEAD")

# No datasets pin: Kaggle now runs Python 3.13, where datasets 2.21's dependencies break.
# Audio is decoded by iqra.data (soundfile on the raw bytes), so any datasets version works.
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
                   capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("pip install failed:\n" + r.stdout[-1500:] + "\n" + r.stderr[-1500:])
sys.path.insert(0, f"{REPO_DIR}/src")
import warnings
warnings.filterwarnings("ignore")
import iqra.align
import datasets, numpy, pandas
print("python", sys.version.split()[0], "| datasets", datasets.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__)
print("repo commit:", REPO_COMMIT)
print("official scorer:", iqra.align.mdd_eval_dir())


In [ ]:
# ---- Environment, inputs -----------------------------------------------------------------
import json, time
import numpy as np
import pandas as pd
import torch, lightgbm
from datasets import load_dataset, load_from_disk
from iqra import data, posteriors, scorer, verifier
from iqra.candidates import candidate_features
from iqra.config import Config, find_cache_root, find_input
from iqra.vocab import Vocab

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
except Exception as e:
    raise RuntimeError("HF_TOKEN secret is required for the gated Iqra_Extra_IS26") from e

cfg = Config()
cfg.paths.cache_root = find_cache_root()
SPLITS_DIR = find_input("splits.json", what="the NB11 output")
POST_DIR = find_input("post_stage1_extra_dev.pt", what="the NB14 output")
OUT = "/kaggle/working/nb15_t3"
os.makedirs(OUT, exist_ok=True)
vocab = Vocab.load(cfg.paths.vocab)
TOK2ID, ID2TOK, BLANK = vocab.tok2id, vocab.id2tok, vocab.blank_id
splits = data.load_splits(f"{SPLITS_DIR}/splits.json")
truth_dev = pd.read_csv(f"{SPLITS_DIR}/truth_extra_dev.csv", keep_default_na=False)
report = {"repo_commit": REPO_COMMIT, "lightgbm": lightgbm.__version__, "torch": torch.__version__}
print("splits:", SPLITS_DIR, "\ncaches:", POST_DIR, "| lightgbm", lightgbm.__version__)


In [ ]:
# ---- Caches and labels (what was actually said) ------------------------------------------
caches, metas = {}, []
for split in ("extra_dev", "fold_a", "fold_b", "iqra_dev_slice"):
    meta, clips = posteriors.read_cache(f"{POST_DIR}/post_stage1_{split}.pt")
    caches[split] = clips
    metas.append(meta)
posteriors.check_compatible(metas)
assert metas[0]["vocab_hash"] == vocab.hash

extra = load_dataset(cfg.datasets.extra, split="train", token=HF_TOKEN)
ex_ids = [str(x) for x in list(extra["id"])]
assert ex_ids == splits["extra"]["ids_by_row"]
ex_ref = dict(zip(ex_ids, [str(x) for x in list(extra["phoneme_ref"])]))
ex_mis = dict(zip(ex_ids, [str(x) for x in list(extra["phoneme_mis"])]))
ex_text = dict(zip(ex_ids, [str(x) for x in list(extra["sentence"])]))
dev = load_from_disk(cfg.paths.train_arrow)["dev"]
dev_phon = list(dev["phoneme_ref"])
dev_ref = {i: str(dev_phon[r]) for i, r in zip(splits["iqra_dev_slice"]["ids"], splits["iqra_dev_slice"]["rows"])}

def expected_said(split, cid):
    if split == "iqra_dev_slice":            # correct speech: said == canonical, every flag is a false alarm
        return dev_ref[cid].split(), dev_ref[cid].split()
    return ex_ref[cid].split(), ex_mis[cid].split()

for split, clips in caches.items():
    assert set(clips) == set(splits["extra"][split] if split != "iqra_dev_slice" else splits["iqra_dev_slice"]["ids"])
print({s: len(c) for s, c in caches.items()})


In [ ]:
# ---- Candidate tables -----------------------------------------------------------------------
t0 = time.time()
tables = {}
for split, clips in caches.items():
    rows = []
    for cid, entry in clips.items():
        exp, said = expected_said(split, cid)
        rows += candidate_features(cid, exp, entry, TOK2ID, ID2TOK, BLANK, said=said)
    df = pd.DataFrame(rows)
    df["split"] = split
    df["sentence"] = [" ".join(expected_said(split, c)[0]) for c in df["clip"]]   # CV group
    tables[split] = df
    df.to_parquet(f"{OUT}/candidates_stage1_{split}.parquet", index=False)
summary = {s: {"candidates": len(d), "real": int(d["label"].sum()), "false_alarms": int((d["label"] == 0).sum()),
               "S": int((d["op"] == "S").sum()), "D": int((d["op"] == "D").sum()),
               "fa_failed_clips": int(d.groupby("clip")["fa_failed"].max().sum())}
           for s, d in tables.items()}
for s, v in summary.items():
    print(f"{s:15s} {v}")
print(f"built in {(time.time() - t0) / 60:.1f} min")
report["candidates"] = summary


In [ ]:
# ---- Train on fold A + fold B + Iqra dev slice; grouped CV by sentence ---------------------
train_df = pd.concat([tables["fold_a"], tables["fold_b"], tables["iqra_dev_slice"]], ignore_index=True)
assert not set(train_df["clip"]) & set(tables["extra_dev"]["clip"]), "Extra-dev leaked into verifier training"
cv = verifier.cross_validate(train_df, list(train_df["sentence"]), cfg.verifier)
n_rounds = int(np.median(cv["best_iterations"]))
print(f"grouped 5-fold CV (by sentence): AUC {cv['auc']:.4f} | precision at recall 0.9: "
      f"{cv['precision_at_recall_0.9']:.4f} | best iterations {cv['best_iterations']} -> {n_rounds}")
# Iqra-dev-slice rows are all negatives from a different source (clip length, sentence style),
# so the pooled AUC partly measures "which dataset" rather than "is the error real".
# The population that matters is Extra candidates: report CV on those rows only.
is_extra = train_df["split"].isin(["fold_a", "fold_b"]).to_numpy()
y_all = train_df["label"].astype(int).to_numpy()
auc_extra = verifier.auc(y_all[is_extra], cv["oof"][is_extra])
p90_extra = verifier.precision_at_recall(y_all[is_extra], cv["oof"][is_extra])
print(f"  on Extra-fold rows only: AUC {auc_extra:.4f} | precision at recall 0.9: {p90_extra:.4f} "
      f"(base rate {y_all[is_extra].mean():.3f})")
booster, enc = verifier.train(train_df, n_rounds, cfg.verifier)
booster.save_model(f"{OUT}/verifier_stage1.txt")
imp = pd.Series(booster.feature_importance("gain"), index=verifier.FEATURES).sort_values(ascending=False)
print("top features (gain):", {k: round(float(v), 1) for k, v in imp.head(10).items()})
report["cv"] = {"auc": round(cv["auc"], 4), "precision_at_recall_0.9": round(cv["precision_at_recall_0.9"], 4),
                "auc_extra_rows": round(auc_extra, 4), "precision_at_recall_0.9_extra_rows": round(p90_extra, 4),
                "base_rate_extra_rows": round(float(y_all[is_extra].mean()), 4),
                "n_rounds": n_rounds, "train_rows": len(train_df), "train_real": int(train_df["label"].sum())}
report["feature_importance_top10"] = {k: round(float(v), 1) for k, v in imp.head(10).items()}


In [ ]:
# ---- tau sweep on Extra-dev (exact per-clip counts), robust tau, top-3 re-scored officially ----
dev_df = tables["extra_dev"]
probs = verifier.predict(booster, enc, dev_df)
heard = {cid: e["tokens"] for cid, e in caches["extra_dev"].items()}
expected = dict(zip(truth_dev["ID"], truth_dev["Reference_phn"].str.split()))
sweep = verifier.sweep_tau(dev_df, probs, truth_dev, heard, list(cfg.verifier.tau_grid))
sweep.to_csv(f"{OUT}/tau_sweep.csv", index=False)
print(sweep.round(4).to_string(index=False))
tau = verifier.robust_tau(sweep, cfg.verifier.robust_tau_band)

def official_for(t, keep_dir=None):
    new = verifier.apply(dev_df, probs, expected, heard, t)
    pred = pd.DataFrame({"ID": list(new), "Prediction": [" ".join(v) for v in new.values()]})
    return scorer.official_score(truth_dev, pred, keep_dir=keep_dir), pred

top3 = sweep.sort_values("f1", ascending=False).head(3)["tau"].tolist()
official_top3 = {t: official_for(t)[0]["f1"] for t in top3}
for t, f in official_top3.items():
    exact = float(sweep.loc[sweep["tau"] == t, "f1"].iloc[0])
    assert round(exact, 4) == f, (t, exact, f)            # exact counts == official scorer
res, pred_v = official_for(tau, keep_dir=f"{OUT}/score_verifier_extra_dev")
pred_v.to_csv(f"{OUT}/pred_verifier_stage1_extra_dev.csv", index=False)
print(f"top-3 tau re-scored officially: {official_top3} | robust tau = {tau}")
report["tau"] = {"robust": tau, "top3_official": official_top3}


In [ ]:
# ---- GO / NO-GO: verifier + stage-1 vs B0, paired sentence bootstrap -------------------------
pred_b0 = pd.DataFrame({"ID": list(heard), "Prediction": [" ".join(v) for v in heard.values()]})
b0 = scorer.official_score(truth_dev, pred_b0)
clusters = dict(zip(truth_dev["ID"], truth_dev["Reference_phn"]))
c_b0, c_v = scorer.per_clip_counts(truth_dev, pred_b0), scorer.per_clip_counts(truth_dev, pred_v)
gain = scorer.paired_bootstrap_gain(c_b0, c_v, clusters)
row6 = scorer.bootstrap_f1_interval(c_v, clusters)
GO = gain["gain"] >= 0.02 and gain["excludes_zero"] and gain["lo"] > 0
print(f"B0      : P={b0['precision']} R={b0['recall']} F1={b0['f1']}")
print(f"row 6   : P={res['precision']} R={res['recall']} F1={res['f1']}  (tau={tau}; 90% CI "
      f"[{row6['lo']:.4f}, {row6['hi']:.4f}])")
print(f"gain    : {gain['gain']:+.4f}  90% paired CI over {gain['n_clusters']} sentences "
      f"[{gain['lo']:+.4f}, {gain['hi']:+.4f}]")
print(">>> GO" if GO else ">>> NO-GO: stop for a decision")
report["B0"] = {k: b0[k] for k in ("precision", "recall", "f1", "TR", "FR", "FA")}
report["row6"] = {k: res[k] for k in ("precision", "recall", "f1", "TR", "FR", "FA")}
report["row6"]["ci90"] = [round(row6["lo"], 4), round(row6["hi"], 4)]
report["gain"] = {k: (round(v, 4) if isinstance(v, float) else v) for k, v in gain.items()}
report["GO"] = bool(GO)


In [ ]:
# ---- Diagnostics: per error type, flags removed, Quranic share --------------------------------
dev_df = dev_df.assign(prob=probs, kept=probs >= tau)
by_type = []
for t, g in dev_df.groupby("err_type"):
    kept = g[g["kept"]]
    by_type.append({"err_type": t, "candidates": len(g), "real": int(g["label"].sum()),
                    "precision_before": round(float(g["label"].mean()), 3),
                    "kept": len(kept), "precision_after": round(float(kept["label"].mean()), 3) if len(kept) else None,
                    "real_kept": int(kept["label"].sum())})
by_type = pd.DataFrame(by_type).sort_values("candidates", ascending=False)
print(by_type.to_string(index=False))
removed = dev_df[~dev_df["kept"]]
print(f"flags removed: {len(removed)} of {len(dev_df)}; of those, real errors: {int(removed['label'].sum())}")
report["per_error_type"] = by_type.to_dict(orient="records")
report["removed"] = {"removed": len(removed), "of": len(dev_df), "real_removed": int(removed["label"].sum())}
try:
    quran = data.load_tanzil_simple_clean()
    flags = dict(zip(ex_ids, data.quranic_window_flags([ex_text[i] for i in ex_ids], quran)))
    share = {s: round(float(np.mean([flags[i] for i in splits["extra"][s]])), 3)
             for s in ("extra_dev", "fold_a", "fold_b")}
    share["all"] = round(float(np.mean(list(flags.values()))), 3)
    print("Quranic share (3-word runs, error markers repaired):", share)
    print("not matched, examples:", list(dict.fromkeys(ex_text[i] for i in ex_ids if not flags[i]))[:6])
except Exception as e:
    share = f"unknown: {type(e).__name__}: {e}"
report["quranic_share"] = share
with open(f"{OUT}/nb15_t3_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=1, default=str)


In [ ]:
# ---- PASTE BACK ----------------------------------------------------------------------------
print(json.dumps(report, ensure_ascii=False, indent=1, default=str))
